This notebook introduces a minimal attention_based neural language learner trained on an artificial language with singular and plural number particles.

The experiment investigates whether the learner can generalise a distributional regularity to a noun-number combination absent from its training data.

Step 1: Define examples that belong to our artificial language

In [ ]:
examples = ["one dax pa", "two dax ti"]

In [ ]:
print(examples)

Step 2: Add examples of artificial language to the training data

In [ ]:
examples.extend(["one mit pa", "two mit ti"])

In [ ]:
print(examples)

Step 3: Add an example containing a new noun (singular only) to test generalisation later

In [ ]:
examples.append("one wug pa")

In [ ]:
print(examples)

Step 4: Count examples in the training data

In [ ]:
len(examples)

Step 5: Split one example into individual words (tokens)

In [ ]:
examples[1].split()

Step 6: Tokenize all training examples and display the result

In [ ]:
for sentence in examples:
  print(sentence.split())

Step 7: Store the tokenized examples

In [ ]:
tokens = []
for sentence in examples:
  tokens.append(sentence.split())

In [ ]:
print(tokens)

Step 8: Collect all tokens into a single list (words)

In [ ]:
words = []

for sentence in tokens:
  words.extend(sentence)

In [ ]:
print(words)

Step 9: Create our vocabulary (unique tokens)


In [ ]:
vocabulary = set(words)

In [ ]:
print(vocabulary)

In [ ]:
vocabulary = sorted(vocabulary)

In [ ]:
print(vocabulary)

Step 10: Assign a unique numerical ID to each token


In [ ]:
for number, word in enumerate(vocabulary):
  print(number, word)

In [ ]:
word_to_number = {}

for number, word in enumerate(vocabulary):
  word_to_number[word] = number

In [ ]:
print(word_to_number)

Step 11: Encode our training examples as sequences of token IDs

In [ ]:
numbers = []

for word in tokens[0]:
  numbers.append(word_to_number[word])

In [ ]:
print(numbers)

In [ ]:
numerical_examples = []

for sentence in tokens:
  numbers = []
  for word in sentence:
    numbers.append(word_to_number[word])
  numerical_examples.append(numbers)

In [ ]:
print(numerical_examples)

Step 12: Import PyTorch nad convert numerical examples into tensors

In [ ]:
import torch
torch.manual_seed(42)

In [ ]:
first_sentence = torch.tensor(numerical_examples[0])

In [ ]:
print(first_sentence)

In [ ]:
tensors = []

for numbers in numerical_examples:
  tensors.append(torch.tensor(numbers))

In [ ]:
print(tensors)

Step 13: Create word embeddings (4 embedding dimensions per token)

In [ ]:
embedding = torch.nn.Embedding(len(vocabulary), 4)

In [ ]:
print(embedding(tensors[0]))

In [ ]:
print(embedding(tensors[0]).shape)

Step 14: Represent word order using positional embeddings

In [ ]:
positions = torch.arange(3)
print(positions)

In [ ]:
position_embedding = torch.nn.Embedding(3, 4)

In [ ]:
print(position_embedding(positions))

In [ ]:
combined = embedding(tensors[0]) + position_embedding(positions)

In [ ]:
print(combined.shape)

Step 15: Prepare input for attention

In [ ]:
input_batch = combined.unsqueeze(0)

In [ ]:
print(input_batch.shape)

Step 16: Add self-attention mechanism and mask (preventing access to future tokens)

In [ ]:
attention = torch.nn.MultiheadAttention(embed_dim=4, num_heads=1, batch_first=True)

In [ ]:
mask = torch.tensor([
    [False, True, True],
    [False, False, True],
    [False, False, False]
])

In [ ]:
print(mask)

In [ ]:
output, weights = attention(
    input_batch,
    input_batch,
    input_batch,
    attn_mask=mask
)

In [ ]:
print(output.shape)

In [ ]:
print(weights)

Step 17: Use attention to predict the next word

In [ ]:
prediction = torch.nn.Linear(4, len(vocabulary))

In [ ]:
scores = prediction(output)

In [ ]:
print(scores.shape)

In [ ]:
print(scores[0, 1, word_to_number["pa"]])
print(scores[0, 1, word_to_number["ti"]])

In [ ]:
probabilities = torch.softmax(scores[0, 1], dim=0)

In [ ]:
print(probabilities[word_to_number["pa"]])
print(probabilities[word_to_number["ti"]])

Step 18: Calculate the model's prediction error (loss)

In [ ]:
correct_word = word_to_number["pa"]
loss = -torch.log(probabilities[correct_word])
print(loss.item())

In [ ]:
loss.backward() # backpropagation

In [ ]:
parameters = list(embedding.parameters())
parameters += list(position_embedding.parameters())
parameters += list(attention.parameters())
parameters += list(prediction.parameters())

In [ ]:
optimizer = torch.optim.SGD(parameters, lr=0.1)

In [ ]:
optimizer.step()

In [ ]:
combined = embedding(tensors[0]) + position_embedding(positions)

input_batch = combined.unsqueeze(0)

output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask)

scores = prediction(output)

In [ ]:
probabilities = torch.softmax(scores[0, 1], dim=0)

print(probabilities[word_to_number["pa"]].item())

In [ ]:
new_loss = -torch.log(probabilities[word_to_number["pa"]])

print(new_loss.item())

Step 19: Prepare all examples for training

In [ ]:
for sentence in tensors:
  print(sentence[:2], "->", sentence[2])

In [ ]:
for sentence in tensors:
  input_words = sentence[:2]
  combined = embedding(input_words) + position_embedding(positions[:2])
  print(combined.shape)

In [ ]:
mask_2 = torch.tensor([
    [False, True],
    [False, False]
])

In [ ]:
print(mask_2.shape)

In [ ]:
for sentence in tensors:
  input_words = sentence[:2]
  combined = embedding(input_words) + position_embedding(positions[:2])
  input_batch = combined.unsqueeze(0)
  output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask_2)
  scores = prediction(output)
  print(scores[0, 1].shape)

Step 20: Training the learner

In [ ]:
print(scores[0, 1, word_to_number["pa"]])
print(scores[0, 1, word_to_number["ti"]])

In [ ]:
for sentence in tensors:
  input_words = sentence[:2]
  combined = embedding(input_words) + position_embedding(positions[:2])
  input_batch = combined.unsqueeze(0)
  output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask_2)
  scores = prediction(output)
  print(scores[0, 1].shape)

  probabilities = torch.softmax(scores[0, 1], dim=0)
  loss = -torch.log(probabilities[sentence[2]])
  print(loss.item())

In [ ]:
for epoch in range(10):
  total_loss = 0

  for sentence in tensors:
    optimizer.zero_grad()

    input_words = sentence[:2]
    combined = embedding(input_words) + position_embedding(positions[:2])
    input_batch = combined.unsqueeze(0)
    output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask_2)
    scores = prediction(output)

    probabilities = torch.softmax(scores[0, 1], dim=0)
    loss = -torch.log(probabilities[sentence[2]])

    loss.backward()
    optimizer.step()
    total_loss += loss.item()


  print("Epoch:", epoch + 1, "Average loss:", total_loss / len(tensors))

Step 21: Testing generalisation

In [ ]:
test_words = torch.tensor([word_to_number["two"], word_to_number["wug"]])

In [ ]:
print(test_words)

In [ ]:
combined = embedding(test_words) + position_embedding(positions[:2])
input_batch = combined.unsqueeze(0)

In [ ]:
print(input_batch.shape)

In [ ]:
output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask_2)
scores = prediction(output)

In [ ]:
print(scores.shape)

In [ ]:
probabilities = torch.softmax(scores[0, 1], dim=0)

print("P(pa):", probabilities[word_to_number["pa"]].item())
print("P(ti):", probabilities[word_to_number["ti"]].item())

* The learner encountered "one dax pa" and "two dax ti"
* It encountered "one mit pa" and "two mit ti"
* It encountered "one wug pa"
* It never encountered two wug ti

After training, where does the model assign its probability following two wug, pa or ti? Is there evidence of generalisation beyond the specific examples in the training data?

Step 22: Final control test

In [ ]:
test_words = torch.tensor([word_to_number["one"], word_to_number["wug"]])

In [ ]:
combined = embedding(test_words) + position_embedding(positions[:2])

input_batch = combined.unsqueeze(0)

output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask_2)

scores = prediction(output)

probabilities = torch.softmax(scores[0, 1], dim=0)

print("P(pa):", probabilities[word_to_number["pa"]].item())
print("P(ti):", probabilities[word_to_number["ti"]].item())